In [5]:
python -m pip install pandas matplotlib ipykernel

SyntaxError: invalid syntax (2789312830.py, line 1)

In [6]:
import pandas as pd
df = pd.read_csv("../data/Car_Insurance_Claim.csv")
df.head(3)

,ID,AGE,GENDER,RACE,DRIVING_EXPERIENCE,EDUCATION,INCOME,CREDIT_SCORE,VEHICLE_OWNERSHIP,VEHICLE_YEAR,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,VEHICLE_TYPE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,OUTCOME
0,569520,65+,female,majority,0-9y,high school,upper class,0.629027,1.0,after 2015,0.0,1.0,10238,12000.0,sedan,0,0,0,0.0
1,750365,16-25,male,majority,0-9y,none,poverty,0.357757,0.0,before 2015,0.0,0.0,10238,16000.0,sedan,0,0,0,1.0
2,199901,16-25,female,majority,0-9y,high school,working class,0.493146,1.0,before 2015,0.0,0.0,10238,11000.0,sedan,0,0,0,0.0


#Phase 1 - Business Understanding

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/Car_Insurance_Claim.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 10000
Columns: 19


,ID,AGE,GENDER,RACE,DRIVING_EXPERIENCE,EDUCATION,INCOME,CREDIT_SCORE,VEHICLE_OWNERSHIP,VEHICLE_YEAR,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,VEHICLE_TYPE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,OUTCOME
0,569520,65+,female,majority,0-9y,high school,upper class,0.629027,1.0,after 2015,0.0,1.0,10238,12000.0,sedan,0,0,0,0.0
1,750365,16-25,male,majority,0-9y,none,poverty,0.357757,0.0,before 2015,0.0,0.0,10238,16000.0,sedan,0,0,0,1.0
2,199901,16-25,female,majority,0-9y,high school,working class,0.493146,1.0,before 2015,0.0,0.0,10238,11000.0,sedan,0,0,0,0.0
3,478866,16-25,male,majority,0-9y,university,working class,0.206013,1.0,before 2015,0.0,1.0,32765,11000.0,sedan,0,0,0,0.0
4,731664,26-39,male,majority,10-19y,none,working class,0.388366,1.0,before 2015,0.0,0.0,32765,12000.0,sedan,2,0,1,1.0


## 1.1 Context and problem

This project looks at car insurance. When a Insurance company sells a policy, They have to guess how likly a customer is likly to make a claim. If it guess too low it can lose money from paying out claims but if it guess too high for a customer , The customer may go somewhere else and not get insurance of the insurance company and go looking else where

The Specific problem we have is :  **Using information the insurer has when the policy starts(driver details,vehicle details,driving history and age),Also if we can predict whether a customer will make a claim or not ?**

The dataset im using is a Car insurance Claim dataset from kaggle , It has over 10,000 customer and 19 columns. The 'OUTCOME' column shows if the customer has made a claim or not . 1 being shown as 'has made a claim' and 0 shown as 'Not made a claim'


In [ ]:
# 0 no claim, 1 Has a claim
counts = df["OUTCOME"].value_counts()
percent = df["OUTCOME"].value_counts(normalize=True) * 100

pd.DataFrame({"Customers": counts, "Percent": percent.round(1)})

,Customers,Percent
OUTCOME,,
0.0,6867,68.7
1.0,3133,31.3


The table above shows us that 31.3% of customers have made a claim and the other 68.7% did not. So this shows that claims are in the smaller group 


## 1.2 Stakeholders and users

|Who | Role |
|---|---|
|Customer| Supply data and are affected by the result  as this can change their price or weather they can get insured or not |
|Pricing team| These are the main users , they will use the prediction to help set the risk leve and price of the policy given |
|Claims Department| Can use it to plan for how many claims to expect |
|Company managment|Want fewer unexpected losses and fair consistent pricing |

## 1.3 Business Objective
The business would want to find the higher risk customers before a policy is sold to them so it can price them correctly and keep the price consistent between customers. This model would support the underwriters decision, not replace them

## 1.4 Analytics objective
This is a binary Classification task. The taget is 'OUTCOME' (0 = no claim, 1 = has a claim). The inputs are in other columns
The model should give a prediction 'claim or no claim' and a probability of claiming so customers can be ranked as low risk or high risk
This will allow the insures to give prices appropriatly to the given customers



## 1.5 Success criteria

A model that always predicts 'no claim' would already be 68.7% accurate, so accuracy on its own is not a good mesaure here

**Technical criteria**
- Clearly beat the 68.7% baseline.
- Recall for the claim class (1) of at least 0.70, so most customers who do claim are caught.
- ROC-AUC of at least 0.80 on the test set.
- Similar results on the training and test sets, to show the model is not overfitting.

**Practical criteria**
- An underwriter can understand which factors drive the prediction (e.g. past accidents, driving experience, mileage).
- The model does not use race, and its results are checked so it does not treat groups unfairly.
- The results are clear enough to support a pricing decision, not just a number.



In [7]:
# Columns with missing values
missing = df.isnull().sum()

print("\n\n")
print(df["RACE"].value_counts())
print(df["GENDER"].value_counts())




RACE
majority    9012
minority     988
Name: count, dtype: int64
GENDER
female    5010
male      4990
Name: count, dtype: int64


## 1.7 Initial project plan

| CRISP-DM phase | What I will do |
|---|---|
| 1. Business Understanding | Define the problem, stakeholders, objectives and success criteria  |
| 2. Data Understanding | Explore the data: size, data types, missing values, class balance, charts of each feature against `OUTCOME` |
| 3. Data Preparation | Fill or remove missing values, drop `ID` and `RACE`, encode categories, split into train and test sets |
| 4. Modelling | Start with Descion Tree and Random Forrest |
| 5. Evaluation | Compare models using recall, precision, F1, ROC-AUC and a confusion matrix, against the success criteria in 1.5 |
| 6. Deployment | Explain how the model could be used by underwriters, its limits, and next steps |

# Phase 2 : Data Understanding

## Initial Data Collection report

- **Source :** Kaggle - https://www.kaggle.com/datasets/sagnik1511/car-insurance-data
- **Publisher :** Sagnik Roy
- **Date Accessed :** 2nd November 2026
- **License :** License is Unknown
- **How Data was Collected :** Data was Collected Through car insurance companies and some of the data is generated by the author its self
- **Population, Time and Place** There is over 10,000 Customers in the dataSet but as for Dates none are given by the author but has been updated last 5 years ago, Data is in the US as PostCodes determine they are 
- **Grouping and Rounding :** Age,Driving expierience and income are in given groups and not exact valuesm Annual Milage is rounded to teh nearest 1,000.
- **Files :** This is in 1 file : 'Car_Insurance_Claim.csv' 





In [ ]:
df.info()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Lowest and highest value for each number column
df.describe().loc[["min", "max"]].round(2)

#This will show the Data Types of each of these
for col in ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"]:
    print(col, ":", df[col].unique())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   10000 non-null  int64  
 1   AGE                  10000 non-null  str    
 2   GENDER               10000 non-null  str    
 3   RACE                 10000 non-null  str    
 4   DRIVING_EXPERIENCE   10000 non-null  str    
 5   EDUCATION            10000 non-null  str    
 6   INCOME               10000 non-null  str    
 7   CREDIT_SCORE         9018 non-null   float64
 8   VEHICLE_OWNERSHIP    10000 non-null  float64
 9   VEHICLE_YEAR         10000 non-null  str    
 10  MARRIED              10000 non-null  float64
 11  CHILDREN             10000 non-null  float64
 12  POSTAL_CODE          10000 non-null  int64  
 13  ANNUAL_MILEAGE       9043 non-null   float64
 14  VEHICLE_TYPE         10000 non-null  str    
 15  SPEEDING_VIOLATIONS  10000 non-null  int64  
 16

## 2.2 Data Description Report

This dataset has over 10,000 Rows and 19 columns , Each row is only 1 customer 'CREDIT_SCORE' and 'ANNUAL_MILEAGE' have fewer than 10,000 values so some have missing data

### Data dictionary

| Column | Meaning | Data type | Kind of variable | Unit | Valid values / range | Role |
|---|---|---|---|---|---|---|
| ID | Customer ID | int64 | Identifier | – | Unique for every customer | Identifier – drop |
| AGE | Age group | str | Category (ordered) | Years, grouped | 16-25, 26-39, 40-64, 65+ | Input |
| GENDER | Gender | str | Category (unordered) | – | female, male | Input |
| RACE | Race group | str | Category (unordered) | – | majority, minority | Sensitive – drop |
| DRIVING_EXPERIENCE | Years driving | str | Category (ordered) | Years, grouped | 0-9y, 10-19y, 20-29y, 30y+ | Input |
| EDUCATION | Highest education | str | Category (ordered) | – | none, high school, university | Input |
| INCOME | Income group | str | Category (ordered) | – | poverty, working class, middle class, upper class | Input |
| CREDIT_SCORE | Credit score | float64 | Number | Score (scale not explained) | 0.05 to 0.96 | Input |
| VEHICLE_OWNERSHIP | Owns the car | float64 | Yes/No | – | 0 = no, 1 = yes | Input |
| VEHICLE_YEAR | When the car was made | str | Category (ordered) | – | before 2015, after 2015 | Input |
| MARRIED | Married | float64 | Yes/No | – | 0 = no, 1 = yes | Input |
| CHILDREN | Has children | float64 | Yes/No | – | 0 = no, 1 = yes | Input |
| POSTAL_CODE | Area code | int64 | Category (unordered) | – | 10238, 21217, 32765, 92101 | Input – check |
| ANNUAL_MILEAGE | Distance driven per year | float64 | Number | Miles (assumed) | 2,000 to 22,000 | Input |
| VEHICLE_TYPE | Type of car | str | Category (unordered) | – | sedan, sports car | Input |
| SPEEDING_VIOLATIONS | Speeding fines | int64 | Count | Number of fines | 0 to 22 | Input |
| DUIS | Drink-driving offences | int64 | Count | Number of offences | 0 to 6 | Input |
| PAST_ACCIDENTS | Past accidents | int64 | Count | Number of accidents | 0 to 15 | Input |
| OUTCOME | Made a claim | float64 | Yes/No | – | 0 = no claim, 1 = claim | **Target** |

**Ordered** categories have a natural order (16-25 is younger than 65+). **Unordered** categories are just labels (female and male). This decides how each column is turned into numbers in Data Preparation.


## Identifiersm Inputs and Target

- **Identifier :** ID is unique for each customer and has no link or corearlation to having a claim so that would be dropped
- **Target** 'OUTCOME', Weather the customer made a claim (1) or customer did not make a claim (0)
- **Candinate Inputs :* 17 columns are all the things a insurer would know when a policy starts* 



### How the variables were measured
- `AGE`, `DRIVING_EXPERIENCE` and `INCOME` are given in groups, not exact values, so some detail is lost.
- `CREDIT_SCORE` is between 0 and 1, but the dataset doesn't explain how it was calculated.
- `ANNUAL_MILEAGE` is rounded to the nearest 1,000. The unit isn't stated. it could be asssumed as in Miles as some of the Postal Codes were American
- `VEHICLE_OWNERSHIP`, `MARRIED`, `CHILDREN` and `OUTCOME` only hold 0 or 1 but are stored as decimals.
- `POSTAL_CODE` is stored as a number but is really a category. A bigger code doesn't mean anything.

In [16]:
df.info()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Lowest and highest value for each number column
df.describe().loc[["min", "max"]].round(2)



#This will show the Data Types of each of these
for col in ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"]:
    print(col, ":", df[col].unique())


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   10000 non-null  int64  
 1   AGE                  10000 non-null  str    
 2   GENDER               10000 non-null  str    
 3   RACE                 10000 non-null  str    
 4   DRIVING_EXPERIENCE   10000 non-null  str    
 5   EDUCATION            10000 non-null  str    
 6   INCOME               10000 non-null  str    
 7   CREDIT_SCORE         9018 non-null   float64
 8   VEHICLE_OWNERSHIP    10000 non-null  float64
 9   VEHICLE_YEAR         10000 non-null  str    
 10  MARRIED              10000 non-null  float64
 11  CHILDREN             10000 non-null  float64
 12  POSTAL_CODE          10000 non-null  int64  
 13  ANNUAL_MILEAGE       9043 non-null   float64
 14  VEHICLE_TYPE         10000 non-null  str    
 15  SPEEDING_VIOLATIONS  10000 non-null  int64  
 16

In [18]:
df.describe().round(2)

,ID,CREDIT_SCORE,VEHICLE_OWNERSHIP,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,OUTCOME
count,10000.00,9018.00,10000.00,10000.0,10000.00,10000.00,9043.00,10000.00,10000.00,10000.00,10000.00
mean,500521.91,0.52,0.70,0.5,0.69,19864.55,11697.00,1.48,0.24,1.06,0.31
std,290030.77,0.14,0.46,0.5,0.46,18915.61,2818.43,2.24,0.55,1.65,0.46
min,101.00,0.05,0.00,0.0,0.00,10238.00,2000.00,0.00,0.00,0.00,0.00
25%,249638.50,0.42,0.00,0.0,0.00,10238.00,10000.00,0.00,0.00,0.00,0.00
50%,501777.00,0.53,1.00,0.0,1.00,10238.00,12000.00,0.00,0.00,0.00,0.00
75%,753974.50,0.62,1.00,1.0,1.00,32765.00,14000.00,2.00,0.00,2.00,1.00
max,999976.00,0.96,1.00,1.0,1.00,92101.00,22000.00,22.00,6.00,15.00,1.00
